# Intermediate: A Simple Rules-Based GenAI Use-Case Recommender

**Sub-topic 1.2 — AI Landscape and Industry Use Cases**

Given an industry and a free-text problem description, this recommends the closest-matching documented use case by keyword overlap. It is deliberately simple (no ML, no embeddings) — the point is to practice the "structured data + query logic = decision support" pattern before you rebuild this same idea with an LLM in Track 3 (semantic search / RAG will replace the crude keyword matching used here).

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("industry_use_cases.csv")
df = pd.read_csv(DATA_PATH)
df.head(3)

,industry,use_case,genai_capability,modality,maturity
0,Healthcare,Summarizing clinical notes,summarization,text,Established
1,Healthcare,Drafting patient communication,drafting,text,Established
2,Healthcare,Medical image triage assistance,classification,image,Emerging


## The recommender function

Scores each candidate use case by how many words it shares with the problem description, then breaks ties by preferring `Established` use cases (a safer real-world bet than an `Emerging` one).

In [2]:
def recommend(df: pd.DataFrame, industry: str, problem_description: str) -> pd.DataFrame:
    """Return use cases for `industry`, ranked by keyword overlap with
    `problem_description`, most relevant first."""
    candidates = df[df["industry"].str.lower() == industry.lower()].copy()
    if candidates.empty:
        return candidates

    problem_words = set(problem_description.lower().split())

    def score(row: pd.Series) -> int:
        use_case_words = set(row["use_case"].lower().split())
        capability_words = set(row["genai_capability"].lower().split())
        return len(problem_words & (use_case_words | capability_words))

    candidates["match_score"] = candidates.apply(score, axis=1)
    # Prefer established use cases when scores tie -- a safer real-world bet.
    candidates["maturity_rank"] = (candidates["maturity"] == "Established").astype(int)
    return candidates.sort_values(
        by=["match_score", "maturity_rank"], ascending=[False, False]
    )


def print_recommendation(df: pd.DataFrame, industry: str, problem: str) -> None:
    print(f"\nIndustry: {industry}")
    print(f"Problem:  {problem!r}")
    results = recommend(df, industry, problem)
    if results.empty:
        print("  No use cases documented for this industry.")
        return
    top = results.iloc[0]
    print(f"  -> Best match: {top['use_case']} ({top['genai_capability']}, "
          f"{top['modality']}, {top['maturity']}, score={top['match_score']})")
    if len(results) > 1:
        print("  Other candidates:")
        for _, row in results.iloc[1:3].iterrows():
            print(f"     - {row['use_case']} (score={row['match_score']})")

In [3]:
scenarios = [
    ("Legal", "we need help drafting contracts faster"),
    ("Retail", "our customers want a chatbot for support conversation"),
    ("Software", "we want to automatically review code for bugs"),
    ("Marketing", "we need to generate ad copy at scale"),
]

for industry, problem in scenarios:
    print_recommendation(df, industry, problem)


Industry: Legal
Problem:  'we need help drafting contracts faster'
  -> Best match: Legal brief drafting assistance (drafting, text, Emerging, score=1)
  Other candidates:
     - Contract review and clause extraction (score=0)
     - Case law summarization (score=0)

Industry: Retail
Problem:  'our customers want a chatbot for support conversation'
  -> Best match: Customer service chatbots (conversation, text, Established, score=1)
  Other candidates:
     - Personalized product descriptions (score=0)
     - Virtual try-on image generation (score=0)

Industry: Software
Problem:  'we want to automatically review code for bugs'
  -> Best match: Automated code review (analysis, code, Established, score=2)
  Other candidates:
     - Code generation (score=1)
     - Test case generation (score=0)

Industry: Marketing
Problem:  'we need to generate ad copy at scale'
  -> Best match: Ad copy generation (generation, text, Established, score=2)
  Other candidates:
     - Campaign personalizat

### Note on this approach's limits

This recommender only matches **literal** overlapping words. It would fail on a synonym like *"assemble contracts"* instead of *"draft contracts"* — it has no notion of **meaning**, only string overlap. This exact limitation is why Track 3 introduces embeddings and vector search: representing meaning as vectors lets you match on semantic similarity instead of exact keyword overlap.

### Try it yourself
Modify `recommend()` so it also returns the `maturity` field directly in the printed recommendation, and try a scenario for an industry not in `scenarios` above.